# function process_image_with_shapes() + execution


In [ ]:
"""
Automates the extraction of infrastructure data from a static map image
(in this scriptIMAGE_PATH) and converts it into a gpkg file. It uses a transformation matrix 
(transform_matrix) calibrated via known pairs (pixel_points, geo_points) 
to translate image coordinates into real-world Latitude and Longitude.

With OpenCV specific infrastructure classes are filtered (e.g., Ports, Refineries) 
based on HSV color ranges and geometric shapes (circles, triangles, squares) and transformed
into geospatial points.
"""

import cv2
import numpy as np
import os
import geopandas as gpd
from shapely.geometry import Point

# Input
IMAGE_PATH = 'africa_supply_iea.png'
OUTPUT_GPKG = 'africa_supply_data.gpkg'

# calibration
pixel_points = np.float32([
    [90, 235],   # Boa Vista
    [848, 1234], # Capo di Buona Speranza 
    [1405, 318] # Corno d'Africa 
]) 

geo_points = np.float32([
    [-22.8182, 16.0953], 
    [20.0086, -34.8322],
    [51.2284, 11.8322] 
])

transform_matrix = cv2.getAffineTransform(pixel_points, geo_points)

# functions
def pixel_to_geo(x, y, matrix):
    point = np.array([x, y, 1.0])
    geo = np.dot(matrix, point)
    return geo[1], geo[0]

def process_image_with_shapes(image_path, output_gpkg):
    if not os.path.exists(image_path):
        print(f"Errore: Il file '{image_path}' non è stato trovato.")
        return

    img = cv2.imread(image_path)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # shape recognition
    target_classes = {
        'Ports': (
            np.array([40, 100, 100]), np.array([80, 255, 255]), 'circle'
        ),
        'Storage facilities': (
            np.array([10, 150, 150]), np.array([25, 255, 255]), 'triangle'
        ),
        'Production clusters': (
            np.array([140, 100, 100]), np.array([170, 255, 255]), 'circle'
        ),
        'Refineries': (
            np.array([110, 20, 50]), np.array([160, 150, 200]), 'square' 
        )
    }

    estratti = []

    for category, (lower, upper, target_shape) in target_classes.items():
        mask = cv2.inRange(hsv, lower, upper)
        contorni, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for contorno in contorni:
            
            if cv2.contourArea(contorno) > 10:
                
                perimetro = cv2.arcLength(contorno, True)
                approssimazione = cv2.approxPolyDP(contorno, 0.04 * perimetro, True)
                vertici = len(approssimazione)

                shape_match = False
                if target_shape == 'triangle' and vertici == 3:
                    shape_match = True
                elif target_shape == 'square' and vertici == 4:
                    shape_match = True
                elif target_shape == 'circle' and vertici > 4:
                    shape_match = True

                if shape_match:
                    M = cv2.moments(contorno)
                    if M["m00"] != 0:
                        cX = int(M["m10"] / M["m00"])
                        cY = int(M["m01"] / M["m00"])
                        
                        lat, lon = pixel_to_geo(cX, cY, transform_matrix)
                        
                        estratti.append({
                            'Category': category,
                            'Pixel_X': cX,
                            'Pixel_Y': cY,
                            'geometry': Point(lon, lat)
                        })

    if not estratti:
        print("No recognized points.")
        return

    gdf = gpd.GeoDataFrame(estratti, crs="EPSG:4326")
    gdf.to_file(output_gpkg, driver="GPKG")
    print(f"Completed. {len(estratti)} points validated and saved in '{output_gpkg}'.")

# execution
if __name__ == "__main__":
    process_image_with_shapes(IMAGE_PATH, OUTPUT_GPKG)

Estrazione completata. 90 punti geometricamente validati salvati in 'africa_supply_data.gpkg'.


# function manual_review() + execution
!!! WARNING: SOMETIMES AFTER SOME POINTS IT STARTS SAVING THINGS IN WRONG FIELDS !!!

In [ ]:
"""
!!! WARNING: SOMETIMES AFTER SOME POINTS IT STARTS SAVING THINGS IN WRONG FIELDS!!!

this function opens Google Maps (satellite) to manually check and update approximate locations 
(GPKG_FILE) extracted with process_image_with_shapes().
It focuses only on points that need review.
The user can then type in a new name, coordinates, and notes for that point.
All changes are saved directly back into the original database file, in new fields "Exact_Name", 
"Exact_Lat", "Exact_Lon", and "Notes".
"""

import geopandas as gpd
import pandas as pd
import webbrowser
import os

GPKG_FILE = 'africa_supply_data.gpkg'

def manual_review(file_path):
    if not os.path.exists(file_path):
        print(f"Error: File '{file_path}' not found.")
        return

    print("Loading data...")
    gdf = gpd.read_file(file_path)

    # Initialize columns if they are missing
    for col in ['Exact_Name', 'Exact_Lat', 'Exact_Lon', 'Notes']:
        if col not in gdf.columns:
            gdf[col] = ""

    col_category = 'Category' if 'Category' in gdf.columns else 'category'

    # Filter for points that need review (missing coordinates and not a production cluster)
    pending = gdf[
        (pd.isna(gdf['Exact_Lat']) | (gdf['Exact_Lat'] == "")) & 
        (gdf[col_category] != "Production clusters")
    ].index

    if len(pending) == 0:
        print("All points have been processed.")
        return

    print(f"Points to review: {len(pending)}")
    print("Available commands: 'exit' to quit, 'skip' to skip the current point.")

    for idx in pending:
        row = gdf.loc[idx]
        category = row.get(col_category, 'Unknown')
        current_name = row.get('Exact_Name', 'N/A')
        
        geom = row.geometry
        if geom is None or geom.is_empty:
            print(f"Index {idx} skipped: Missing geometry.")
            continue
            
        lat, lon = geom.y, geom.x

        print(f"\n--- Index: {idx} | Category: {category} | Name: {current_name} ---")

        # Open Google Maps to the coordinates
        url = f"https://www.google.com/maps/search/?api=1&query={lat},{lon}"
        webbrowser.open(url)

        # User input for new data
        new_name = input(f"Exact Name (Press Enter to keep '{current_name}'): ").strip()
        if new_name.lower() == 'exit': break
        if new_name.lower() == 'skip': continue

        coords_str = input("Coordinates (e.g., '16.06862, -16.54123'): ").strip()
        if coords_str.lower() == 'exit': break
        if coords_str.lower() == 'skip': continue
        
        notes = input("Notes: ").strip()
        if notes.lower() == 'exit': break
        if notes.lower() == 'skip': continue

        # Update the dataframe
        if new_name: 
            gdf.at[idx, 'Exact_Name'] = str(new_name)
        if notes: 
            gdf.at[idx, 'Notes'] = str(notes)
        
        if coords_str:
            try:
                # Clean and split coordinates input
                coords_clean = coords_str.replace(',', ' ')
                parts = [p for p in coords_clean.split() if p]
                
                if len(parts) != 2:
                    raise ValueError(f"Extracted {len(parts)} values instead of 2.")
                
                lat_val = float(parts[0])
                lon_val = float(parts[1])
                
                gdf.at[idx, 'Exact_Lat'] = f"{lat_val:.5f}"
                gdf.at[idx, 'Exact_Lon'] = f"{lon_val:.5f}"

            except ValueError as e:
                print(f"Error: {e} Point NOT saved. Repeat this index next time.")
                continue

        # Save to file after each entry to prevent data loss
        gdf.to_file(file_path, driver="GPKG")
        print("Point saved to file.")

    print("\nSession ended.")

if __name__ == "__main__":
    manual_review(GPKG_FILE)

Extraction complete. 90 points validated and saved to 'africa_supply_data.gpkg'.


# function update_coordinates_from_gmaps() + execution

In [ ]:
"""
This function updates a map file using new coordinates from Exact_Lat and Exact_Lon.
The points are first converted to match the map's original coordinate system.
Finally, it saves the changes back into the GeoPackage file.
"""

import geopandas as gpd

def update_coordinates_from_gmaps(gpkg_path):
    try:
        # Load the GeoPackage
        gdf = gpd.read_file(gpkg_path)
        
        # Check for required columns
        if 'Exact_Lat' not in gdf.columns or 'Exact_Lon' not in gdf.columns:
            print("Error: Columns 'Exact_Lat' or 'Exact_Lon' not found. Script aborted.")
            return

        # Identify rows with data
        mask = gdf['Exact_Lat'].notna() & gdf['Exact_Lon'].notna()
        
        if not mask.any():
            print("No valid coordinates found in Exact_Lat and Exact_Lon. No changes made.")
            return

        # Create new points and define them as WGS 84 (Google Maps default)
        new_points = gpd.GeoSeries.from_xy(
            x=gdf.loc[mask, 'Exact_Lon'], 
            y=gdf.loc[mask, 'Exact_Lat'],
            crs="EPSG:4326"
        )

        # Reproject the new points to match the GeoPackage's native CRS
        new_points_projected = new_points.to_crs(gdf.crs)

        # Overwrite the geometry
        gdf.loc[mask, 'geometry'] = new_points_projected

        # Overwrite the original GeoPackage
        gdf.to_file(gpkg_path, driver="GPKG")
        print(f"Successfully updated and reprojected geometries for {mask.sum()} features.")

    except Exception as e:
        print(f"Failed to process the GeoPackage: {e}")

# Execute the function
update_coordinates_from_gmaps("africa_supply_a.gpkg")

Successfully updated and reprojected geometries for 32 features.


# function add_new_points() + execution

In [ ]:
"""
This function allows adding a new point to the map file by providing coordinates, 
category, name, and notes in an automatic way without the need for QGIS
"""

import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

def add_new_point(gpkg_path):
    # 1. Collect inputs
    coords_input = input("Enter coordinates (Format: Lat, Lon): ")
    category = input("Enter Category: ")
    exact_name = input("Enter Exact_Name: ")
    notes = input("Enter Notes: ")

    # 2. Parse coordinates
    try:
        lat_str, lon_str = coords_input.split(',')
        lat = float(lat_str.strip())
        lon = float(lon_str.strip())
    except ValueError:
        print("Fatal Error: Invalid coordinate format. You must use 'Lat, Lon' separated by a comma. Script aborted.")
        return

    # 3. Process and append
    try:
        # Load the existing GeoPackage
        gdf = gpd.read_file(gpkg_path)
        
        # Create the new point in WGS 84 (EPSG:4326)
        # Note: Shapely Point takes (x, y) which is (Longitude, Latitude)
        new_point = Point(lon, lat)
        
        # Create a single-row GeoDataFrame for the new feature
        new_row = gpd.GeoDataFrame(
            {'Category': [category], 'Exact_Name': [exact_name], 'Notes': [notes]}, 
            geometry=[new_point], 
            crs="EPSG:4326"
        )
        
        # Reproject the new row to match the target GeoPackage
        new_row_projected = new_row.to_crs(gdf.crs)
        
        # Append the new row to the existing data
        # pd.concat is used as gdf.append is deprecated in pandas
        updated_gdf = pd.concat([gdf, new_row_projected], ignore_index=True)
        
        # Overwrite the GeoPackage
        updated_gdf.to_file(gpkg_path, driver="GPKG")
        print("Success: New point added and file overwritten.")
        
    except Exception as e:
        print(f"System Error: Failed to process or save the GeoPackage: {e}")

# Execute the function
add_new_point("africa_supply_a.gpkg")

Success: New point added and file overwritten.
